# State 갱신과 리듀서

노드가 같은 상태 키를 여러 번 반환하면 값은 덮어쓸까요, 누적할까요?

## 실행 환경 준비

Python 3.12 새 커널에서 첫 셀부터 실행합니다. 설치 셀은 이 노트북에 필요한 버전을 고정합니다.

## 패키지 설치와 버전 확인

필요한 패키지를 고정 버전으로 설치합니다. 출력에서 LangGraph 버전 또는 API 키의 존재 여부만 확인합니다.

In [1]:
# 노트북을 독립 실행할 수 있도록 패키지와 환경을 준비합니다.
%pip install -q langgraph==1.2.11 langchain==1.4.0 langchain-openai==1.6.1 python-dotenv==1.2.3 scikit-learn==1.9.0
from importlib.metadata import version
print('Python 패키지 준비 완료')
print('langgraph', version('langgraph'))

Note: you may need to restart the kernel to use updated packages.
Python 패키지 준비 완료
langgraph 1.2.11


## 서로 다른 갱신 규칙 선언

서로 다른 갱신 규칙 선언 단계를 실행합니다. 출력된 상태 값과 코드에서 반환한 키를 서로 대조합니다.

In [2]:
# 서로 다른 갱신 규칙 선언 단계의 연산을 실행하고 반환된 상태 값을 확인합니다.
import operator
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

class UpdateState(TypedDict):
    query: str
    log: Annotated[list[str], operator.add]

def normalize(state: UpdateState) -> dict:
    return {'query': state['query'].strip(), 'log': ['normalize']}

def expand(state: UpdateState) -> dict:
    return {'query': state['query'] + ' 정책', 'log': ['expand']}

## 순차 그래프 컴파일

노드와 엣지를 연결한 뒤 실행 가능한 그래프로 컴파일합니다. 다음 셀에서 이 그래프를 호출합니다.

In [3]:
# 순차 그래프 컴파일 단계의 연산을 실행하고 반환된 상태 값을 확인합니다.
builder = StateGraph(UpdateState)
builder.add_node('normalize', normalize)
builder.add_node('expand', expand)
builder.add_edge(START, 'normalize')
builder.add_edge('normalize', 'expand')
builder.add_edge('expand', END)
graph = builder.compile()
print('갱신 그래프 준비 완료')

갱신 그래프 준비 완료


## 덮어쓰기와 누적 비교

덮어쓰기와 누적 비교 조건으로 값을 하나만 변경합니다. 나란히 출력된 두 결과에서 달라진 항목을 확인합니다.

In [4]:
# 한 조건만 바꾸고 나머지 입력을 유지해 결과 차이를 비교합니다.
result = graph.invoke({'query': '  도서 대출 기간  ', 'log': []})
print('최종 query:', result['query'])
print('누적 log:', result['log'])
assert result['query'] == '도서 대출 기간 정책'
assert result['log'] == ['normalize', 'expand']

최종 query: 도서 대출 기간 정책
누적 log: ['normalize', 'expand']


## 초기 기록을 포함해 다시 실행

초기 기록을 포함해 다시 실행 단계를 실행합니다. 출력된 상태 값과 코드에서 반환한 키를 서로 대조합니다.

In [5]:
# 초기 기록을 포함해 다시 실행 단계의 연산을 실행하고 반환된 상태 값을 확인합니다.
second = graph.invoke({'query': '운영 시간', 'log': ['input']})
print(second)
assert second['log'] == ['input', 'normalize', 'expand']

{'query': '운영 시간 정책', 'log': ['input', 'normalize', 'expand']}
